# Notebook 04 — RAGAS Evaluation for RAG Systems

## Objectives
- Understand RAGAS metrics: faithfulness, answer relevancy, context recall
- Build lightweight proxy evaluators without LLM calls
- Run evaluation on a golden dataset
- Use `day5.evaluation` for production monitoring
- Implement drift detection to catch quality degradation over time

## 1. Manual faithfulness check

In [ ]:
def simple_faithfulness(answer, context):
    """Is the answer supported by the context?"""
    answer_words  = set(answer.lower().split())
    context_words = set(context.lower().split())
    stop = {"the","a","an","is","in","of","and","to","it"}
    content = answer_words - stop
    if not content:
        return 0.0
    overlap = content & context_words
    return len(overlap) / len(content)

answer  = "BM25 is a ranking algorithm used in search"
context = "BM25 is a probabilistic ranking framework used in information retrieval and search engines"
print(f"Faithfulness: {simple_faithfulness(answer, context):.2f}")

bad_answer = "The weather in Paris is warm today"
print(f"Off-topic answer faithfulness: {simple_faithfulness(bad_answer, context):.2f}")

## RAGAS Metric Definitions

| Metric | Question it answers | Production implementation |
|--------|--------------------|--------------------------|
| **Faithfulness** | Is the answer grounded in the retrieved docs? | LLM checks each claim against contexts |
| **Answer Relevancy** | Does the answer address the question? | Generate N questions from answer, check similarity to original |
| **Context Recall** | Are the right documents retrieved? | LLM checks if ground truth is covered by contexts |
| **Context Precision** | Are retrieved docs all relevant? | LLM scores relevance of each context |

Our proxy implementations use word overlap instead of LLM calls — much faster and free to run.

## 2. Golden dataset evaluation

In [ ]:
# Golden dataset: (question, expected_answer, retrieved_contexts)
golden = [
    {
        "question": "What is BM25?",
        "answer":   "BM25 is a ranking algorithm for information retrieval",
        "contexts": ["BM25 is a probabilistic ranking framework used in information retrieval"]
    },
    {
        "question": "How does hybrid search work?",
        "answer":   "Hybrid search combines BM25 and semantic embeddings",
        "contexts": ["Hybrid search combines keyword and semantic retrieval methods"]
    },
    {
        "question": "What is RRF?",
        "answer":   "Reciprocal Rank Fusion merges ranked lists",
        "contexts": ["Reciprocal Rank Fusion merges multiple ranked lists into one"]
    },
]

for item in golden:
    faith = simple_faithfulness(item["answer"], item["contexts"][0])
    print(f"Q: {item['question'][:40]:40s} | faithfulness: {faith:.2f}")

## 3. Production RAGAS (real library)

In production, install `ragas` and use LLM-backed evaluations:

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

data = Dataset.from_dict({
    "question": ["What is BM25?"],
    "answer":   ["BM25 is a ranking algorithm"],
    "contexts": [["BM25 is used in search engines"]],
    "ground_truth": ["BM25 is a keyword ranking function"]
})

result = evaluate(data, metrics=[faithfulness, answer_relevancy, context_recall])
print(result)  # DataFrame with per-metric scores
```

Our `day5.evaluation` module provides the same interface without LLM API calls.

## 4. Setup sys.path

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

## 5. Using day5.evaluation

In [ ]:
from day5.evaluation import evaluate_rag_response, RAGASResult

result = evaluate_rag_response(
    question="What is hybrid search?",
    answer="Hybrid search combines BM25 and semantic retrieval for better results",
    contexts=["Hybrid search combines keyword and semantic retrieval methods"]
)

print(f"Faithfulness    : {result.faithfulness:.3f}")
print(f"Answer relevancy: {result.answer_relevancy:.3f}")
print(f"Context recall  : {result.context_recall:.3f}")
print(f"Overall         : {result.overall:.3f}")

## 6. RAGMonitor — tracking quality over time

In [ ]:
from day5.evaluation import RAGMonitor

monitor = RAGMonitor(faithfulness_threshold=0.5)

# Simulate 5 production queries
queries = [
    ("What is BM25?",         "BM25 is a ranking algorithm",    ["BM25 is used in search"]),
    ("How does RRF work?",    "RRF merges ranked lists",         ["Reciprocal Rank Fusion merges lists"]),
    ("What is LangGraph?",    "LangGraph builds agent graphs",   ["LangGraph builds stateful workflows"]),
    ("What is hybrid search?","Hybrid combines BM25 + semantic",["Hybrid search combines methods"]),
    ("What is RAGAS?",        "RAGAS evaluates RAG systems",     ["RAGAS evaluates faithfulness"]),
]

for q, a, ctx in queries:
    result = monitor.record(q, a, ctx, tokens=50, latency_ms=120)
    print(f"  Q: {q[:30]:30s} | faith={result.faithfulness:.2f} relevancy={result.answer_relevancy:.2f}")

print()
metrics = monitor.get_metrics()
print(f"Total queries      : {metrics.total_queries}")
print(f"Avg faithfulness   : {metrics.avg_faithfulness}")
print(f"Avg latency (ms)   : {metrics.avg_latency_ms}")
print(f"Below threshold    : {metrics.queries_below_threshold}")

## 7. Drift detection

In [ ]:
from day5.evaluation import RAGMonitor

monitor = RAGMonitor()

# Simulate 15 high-quality historical records
for i in range(15):
    monitor._records.append({
        "query": "q", "answer": "a",
        "faithfulness": 0.85, "answer_relevancy": 0.80,
        "tokens": 100, "latency_ms": 80
    })

# Simulate 10 recent low-quality records (e.g., after a bad model update)
for i in range(10):
    monitor._records.append({
        "query": "q", "answer": "a",
        "faithfulness": 0.25, "answer_relevancy": 0.30,
        "tokens": 100, "latency_ms": 80
    })

drift = monitor.detect_drift(window=10)
print("Drift detection result:")
print(f"  Drifted : {drift['drifted']}")
print(f"  Delta   : {drift['delta']} (historic avg - recent avg)")
print(f"  Action  : {drift['action']}")

## 8. CostTracker

In [ ]:
from day5.evaluation import CostTracker

tracker = CostTracker(model="gpt-4o-mini")

# Simulate 5 API calls
queries_data = [
    (500,  120, "What is BM25?"),
    (800,  200, "Explain hybrid search"),
    (300,  90,  "What is RRF?"),
    (1200, 400, "Detailed LangGraph tutorial"),
    (600,  150, "RAGAS evaluation"),
]

for input_t, output_t, q in queries_data:
    cost = tracker.record(input_t, output_t, q)
    print(f"  {q[:35]:35s} → ${cost:.6f}")

print()
summary = tracker.summary()
print(f"Total cost: ${summary['total_cost_usd']:.6f}")
print(f"Avg cost  : ${summary['avg_cost_usd']:.6f} per query")
print(f"Sessions  : {summary['sessions']}")
print(f"Model     : {summary['model']}")